# LAB2 - Bài tập thêm - Bài 2

## Công thức F-measure

F-measure là trung bình điều hòa (harmonic mean) của độ chính xác (precision) và độ phủ (recall):

$$F\text{-}measure = 2 \times \frac{precision \times recall}{precision + recall}$$

Các bài tập cơ bản gồm: **Yêu cầu cơ bản** (mục III.1 - đánh giá rủi ro hồ sơ vay tín dụng), **Bài 1** (mục IV.1 - doanh số đồ chơi) và **Bài 2** (mục IV.2 - phân tích cảm xúc). Để có precision, recall của từng bài, chương trình tự cài đặt lại các thuật toán ID3, CART và Naïve Bayes với làm trơn Laplace giống cách tính tay trong tài liệu, sau đó tính F-measure.

## Cài đặt các hàm

In [1]:
import math
from collections import Counter
import numpy as np
import pandas as pd
from sklearn import metrics

In [2]:
def entropy(labels):
    n = len(labels)
    return -sum((c / n) * math.log2(c / n) for c in Counter(labels).values())

def gini(labels):
    n = len(labels)
    return 1 - sum((c / n) ** 2 for c in Counter(labels).values())

def split_score(df, attr, target, criterion):
    # ID3: độ lợi thông tin (information gain) - chọn giá trị lớn nhất
    # CART: chỉ số Gini của phép chia - chọn giá trị nhỏ nhất
    n = len(df)
    if criterion == 'id3':
        return entropy(list(df[target])) - sum(len(g) / n * entropy(list(g[target]))
                                               for _, g in df.groupby(attr, sort=False))
    return sum(len(g) / n * gini(list(g[target])) for _, g in df.groupby(attr, sort=False))

def majority(labels):
    return Counter(labels).most_common(1)[0][0]

def build_tree(df, attrs, target, criterion):
    labels = list(df[target])
    if len(set(labels)) == 1:
        return {'leaf': labels[0]}
    cand = [a for a in attrs if df[a].nunique() > 1]
    if not cand:
        return {'leaf': majority(labels)}
    scores = {a: round(split_score(df, a, target, criterion), 10) for a in cand}
    # Khi có nhiều thuộc tính bằng nhau, chọn thuộc tính đứng trước trong bảng dữ liệu
    if criterion == 'id3':
        best = max(cand, key=lambda a: scores[a])
    else:
        best = min(cand, key=lambda a: scores[a])
    node = {'attr': best, 'score': scores[best], 'scores': scores,
            'majority': majority(labels), 'children': {}}
    rest = [a for a in attrs if a != best]
    for v, g in df.groupby(best, sort=False):
        node['children'][v] = build_tree(g, rest, target, criterion)
    return node

def print_tree(node, criterion, indent=''):
    name = 'IG' if criterion == 'id3' else 'Gini'
    for v, child in node['children'].items():
        if 'leaf' in child:
            print(f"{indent}{node['attr']} = {v} -> {child['leaf']}")
        else:
            print(f"{indent}{node['attr']} = {v} -> [{child['attr']}] ({name} = {child['score']:.3f})")
            print_tree(child, criterion, indent + '    ')

def show_tree(tree, criterion):
    name = 'IG' if criterion == 'id3' else 'Gini'
    print(f"Gốc: [{tree['attr']}] ({name} = {tree['score']:.3f})")
    print_tree(tree, criterion, '    ')

def find_ties(node, path='Gốc'):
    # Liệt kê các nút có nhiều thuộc tính cùng giá trị tốt nhất
    if 'leaf' in node:
        return []
    ties = [a for a, v in node['scores'].items() if abs(v - node['score']) < 1e-9]
    res = [(path, ', '.join(ties), node['attr'])] if len(ties) > 1 else []
    for v, child in node['children'].items():
        res += find_ties(child, f"{path} > {node['attr']} = {v}")
    return res

def predict_tree(node, row):
    while 'leaf' not in node:
        v = row[node['attr']]
        if v not in node['children']:
            return node['majority']
        node = node['children'][v]
    return node['leaf']

In [3]:
def naive_bayes_laplace(df, attrs, target, row):
    # p(C) = (số mẫu lớp C + 1) / (tổng số mẫu + số lớp)
    # p(x|C) = (số mẫu lớp C có giá trị x + 1) / (số mẫu lớp C + số giá trị của thuộc tính)
    n = len(df)
    classes = list(dict.fromkeys(df[target]))
    k = len(classes)
    probs = {}
    for c in classes:
        dc = df[df[target] == c]
        p = (len(dc) + 1) / (n + k)
        for a in attrs:
            p *= ((dc[a] == row[a]).sum() + 1) / (len(dc) + df[a].nunique())
        probs[c] = p
    return probs

def predict_bayes(df, attrs, target, test):
    table = pd.DataFrame([naive_bayes_laplace(df, attrs, target, r) for _, r in test.iterrows()])
    table['Dự đoán'] = table.idxmax(axis=1)
    return table

In [4]:
def f_measure(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

def evaluate(y_true, y_pred, classes):
    cm = pd.DataFrame(metrics.confusion_matrix(y_true, y_pred, labels=classes),
                      index=['Thực tế: ' + c for c in classes],
                      columns=['Dự đoán: ' + c for c in classes])
    rows = []
    for c in classes:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        rows.append([c, tp, fp, fn, precision, recall, f_measure(precision, recall)])
    report = pd.DataFrame(rows, columns=['Lớp', 'TP', 'FP', 'FN', 'Precision', 'Recall', 'F-measure'])
    report.loc[len(report)] = ['Trung bình (macro)', '', '', '', report['Precision'].mean(),
                               report['Recall'].mean(), report['F-measure'].mean()]
    return cm, report

summary = []
def add_summary(name, model, report, positive):
    # positive = None: bài toán nhiều lớp, dùng giá trị trung bình macro
    avg = report.iloc[-1]
    pos = avg if positive is None else report[report['Lớp'] == positive].iloc[0]
    summary.append([name, model, positive or '(macro)', pos['Precision'], pos['Recall'],
                    pos['F-measure'], avg['F-measure']])

Lưu ý: khi một lớp không được dự đoán lần nào (TP + FP = 0) thì precision của lớp đó được quy ước bằng 0; khi precision + recall = 0 thì F-measure bằng 0.

## Yêu cầu cơ bản (mục III.1)

Đánh giá rủi ro hồ sơ vay tín dụng. Giống tài liệu, chọn **Rủi ro thấp** là lớp dương.

In [5]:
cols = ['Thu nhập', 'Nghề nghiệp', 'Tình trạng hôn nhân', 'Sở hữu nhà', 'Nguy cơ']
raw = """Thấp|Vận tải|Độc thân|Rồi|Rủi ro cao
Thấp|Vận tải|Độc thân|Chưa|Rủi ro cao
Cao|Vận tải|Độc thân|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Độc thân|Rồi|Rủi ro thấp
Trung bình|Kinh doanh|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Kinh doanh|Đã kết hôn|Chưa|Rủi ro cao
Cao|Kinh doanh|Đã kết hôn|Chưa|Rủi ro thấp
Thấp|Truyền thông|Độc thân|Rồi|Rủi ro cao
Thấp|Kinh doanh|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Đã kết hôn|Rồi|Rủi ro thấp
Thấp|Truyền thông|Đã kết hôn|Chưa|Rủi ro thấp
Cao|Truyền thông|Độc thân|Chưa|Rủi ro thấp
Cao|Vận tải|Đã kết hôn|Rồi|Rủi ro thấp
Trung bình|Truyền thông|Độc thân|Chưa|Rủi ro cao
Cao|Kinh doanh|Độc thân|Rồi|Rủi ro cao"""
data_0 = pd.DataFrame([line.split('|') for line in raw.splitlines()], columns=cols)
attrs_0 = cols[:-1]
target_0 = cols[-1]
data_0

,Thu nhập,Nghề nghiệp,Tình trạng hôn nhân,Sở hữu nhà,Nguy cơ
0,Thấp,Vận tải,Độc thân,Rồi,Rủi ro cao
1,Thấp,Vận tải,Độc thân,Chưa,Rủi ro cao
2,Cao,Vận tải,Độc thân,Rồi,Rủi ro thấp
3,Trung bình,Truyền thông,Độc thân,Rồi,Rủi ro thấp
4,Trung bình,Kinh doanh,Đã kết hôn,Rồi,Rủi ro thấp
5,Trung bình,Kinh doanh,Đã kết hôn,Chưa,Rủi ro cao
6,Cao,Kinh doanh,Đã kết hôn,Chưa,Rủi ro thấp
7,Thấp,Truyền thông,Độc thân,Rồi,Rủi ro cao
8,Thấp,Kinh doanh,Đã kết hôn,Rồi,Rủi ro thấp
9,Trung bình,Truyền thông,Đã kết hôn,Rồi,Rủi ro thấp


Đặng Phan Duy - 6451071010

In [6]:
test_0 = pd.DataFrame([line.split('|') for line in """Trung bình|Vận tải|Độc thân|Rồi
Cao|Truyền thông|Độc thân|Rồi
Thấp|Vận tải|Đã kết hôn|Rồi""".splitlines()], columns=attrs_0)
actual_0 = ['Rủi ro cao', 'Rủi ro thấp', 'Rủi ro thấp']
test_0['Thực tế'] = actual_0
test_0

,Thu nhập,Nghề nghiệp,Tình trạng hôn nhân,Sở hữu nhà,Thực tế
0,Trung bình,Vận tải,Độc thân,Rồi,Rủi ro cao
1,Cao,Truyền thông,Độc thân,Rồi,Rủi ro thấp
2,Thấp,Vận tải,Đã kết hôn,Rồi,Rủi ro thấp


Đặng Phan Duy - 6451071010

### Cây quyết định ID3

In [7]:
tree_id3_0 = build_tree(data_0, attrs_0, target_0, 'id3')
print("Information gain của các thuộc tính tại nút gốc:", tree_id3_0['scores'])
show_tree(tree_id3_0, 'id3')

Information gain của các thuộc tính tại nút gốc: {'Thu nhập': 0.0830074999, 'Nghề nghiệp': 0.013315396, 'Tình trạng hôn nhân': 0.1858051629, 'Sở hữu nhà': 0.019973094}
Gốc: [Tình trạng hôn nhân] (IG = 0.186)
    Tình trạng hôn nhân = Độc thân -> [Thu nhập] (IG = 0.360)
        Thu nhập = Thấp -> Rủi ro cao
        Thu nhập = Cao -> [Nghề nghiệp] (IG = 0.918)
            Nghề nghiệp = Vận tải -> Rủi ro thấp
            Nghề nghiệp = Truyền thông -> Rủi ro thấp
            Nghề nghiệp = Kinh doanh -> Rủi ro cao
        Thu nhập = Trung bình -> [Sở hữu nhà] (IG = 1.000)
            Sở hữu nhà = Rồi -> Rủi ro thấp
            Sở hữu nhà = Chưa -> Rủi ro cao
    Tình trạng hôn nhân = Đã kết hôn -> [Thu nhập] (IG = 0.198)
        Thu nhập = Trung bình -> [Sở hữu nhà] (IG = 0.918)
            Sở hữu nhà = Rồi -> Rủi ro thấp
            Sở hữu nhà = Chưa -> Rủi ro cao
        Thu nhập = Cao -> Rủi ro thấp
        Thu nhập = Thấp -> Rủi ro thấp


Đặng Phan Duy - 6451071010

In [8]:
pd.DataFrame(find_ties(tree_id3_0), columns=['Nút', 'Các thuộc tính bằng nhau', 'Thuộc tính được chọn'])

,Nút,Các thuộc tính bằng nhau,Thuộc tính được chọn
0,Gốc > Tình trạng hôn nhân = Đã kết hôn,"Thu nhập, Sở hữu nhà",Thu nhập


Đặng Phan Duy - 6451071010

In [9]:
pred_id3_0 = [predict_tree(tree_id3_0, r) for _, r in test_0.iterrows()]
cm, report = evaluate(actual_0, pred_id3_0, ['Rủi ro thấp', 'Rủi ro cao'])
print("Dự đoán:", pred_id3_0)
display(cm)
display(report)
add_summary('Yêu cầu cơ bản (mục III.1)', 'Cây ID3', report, 'Rủi ro thấp')

Dự đoán: ['Rủi ro thấp', 'Rủi ro thấp', 'Rủi ro thấp']


,Dự đoán: Rủi ro thấp,Dự đoán: Rủi ro cao
Thực tế: Rủi ro thấp,2,0
Thực tế: Rủi ro cao,1,0


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Rủi ro thấp,2,1,0,0.666667,1.0,0.8
1,Rủi ro cao,0,0,1,0.000000,0.0,0.0
2,Trung bình (macro),,,,0.333333,0.5,0.4


Đặng Phan Duy - 6451071010

### Cây quyết định CART

In [10]:
tree_cart_0 = build_tree(data_0, attrs_0, target_0, 'cart')
print("Gini index của các thuộc tính tại nút gốc:", tree_cart_0['scores'])
show_tree(tree_cart_0, 'cart')

Gini index của các thuộc tính tại nút gốc: {'Thu nhập': 0.4266666667, 'Nghề nghiệp': 0.4711111111, 'Tình trạng hôn nhân': 0.3642857143, 'Sở hữu nhà': 0.4666666667}
Gốc: [Tình trạng hôn nhân] (Gini = 0.364)
    Tình trạng hôn nhân = Độc thân -> [Thu nhập] (Gini = 0.292)
        Thu nhập = Thấp -> Rủi ro cao
        Thu nhập = Cao -> [Nghề nghiệp] (Gini = 0.000)
            Nghề nghiệp = Vận tải -> Rủi ro thấp
            Nghề nghiệp = Truyền thông -> Rủi ro thấp
            Nghề nghiệp = Kinh doanh -> Rủi ro cao
        Thu nhập = Trung bình -> [Sở hữu nhà] (Gini = 0.000)
            Sở hữu nhà = Rồi -> Rủi ro thấp
            Sở hữu nhà = Chưa -> Rủi ro cao
    Tình trạng hôn nhân = Đã kết hôn -> [Thu nhập] (Gini = 0.190)
        Thu nhập = Trung bình -> [Sở hữu nhà] (Gini = 0.000)
            Sở hữu nhà = Rồi -> Rủi ro thấp
            Sở hữu nhà = Chưa -> Rủi ro cao
        Thu nhập = Cao -> Rủi ro thấp
        Thu nhập = Thấp -> Rủi ro thấp


Đặng Phan Duy - 6451071010

In [11]:
pd.DataFrame(find_ties(tree_cart_0), columns=['Nút', 'Các thuộc tính bằng nhau', 'Thuộc tính được chọn'])

,Nút,Các thuộc tính bằng nhau,Thuộc tính được chọn
0,Gốc > Tình trạng hôn nhân = Đã kết hôn,"Thu nhập, Sở hữu nhà",Thu nhập


Đặng Phan Duy - 6451071010

In [12]:
pred_cart_0 = [predict_tree(tree_cart_0, r) for _, r in test_0.iterrows()]
cm, report = evaluate(actual_0, pred_cart_0, ['Rủi ro thấp', 'Rủi ro cao'])
print("Dự đoán:", pred_cart_0)
display(cm)
display(report)
add_summary('Yêu cầu cơ bản (mục III.1)', 'Cây CART', report, 'Rủi ro thấp')

Dự đoán: ['Rủi ro thấp', 'Rủi ro thấp', 'Rủi ro thấp']


,Dự đoán: Rủi ro thấp,Dự đoán: Rủi ro cao
Thực tế: Rủi ro thấp,2,0
Thực tế: Rủi ro cao,1,0


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Rủi ro thấp,2,1,0,0.666667,1.0,0.8
1,Rủi ro cao,0,0,1,0.000000,0.0,0.0
2,Trung bình (macro),,,,0.333333,0.5,0.4


Đặng Phan Duy - 6451071010

### Naïve Bayes (làm trơn Laplace)

In [13]:
bayes_0 = predict_bayes(data_0, attrs_0, target_0, test_0)
bayes_0

,Rủi ro cao,Rủi ro thấp,Dự đoán
0,0.017157,0.011343,Rủi ro cao
1,0.011438,0.023632,Rủi ro thấp
2,0.007625,0.014888,Rủi ro thấp


Đặng Phan Duy - 6451071010

In [14]:
pred_bayes_0 = list(bayes_0['Dự đoán'])
cm, report = evaluate(actual_0, pred_bayes_0, ['Rủi ro thấp', 'Rủi ro cao'])
print("Dự đoán:", pred_bayes_0)
display(cm)
display(report)
add_summary('Yêu cầu cơ bản (mục III.1)', 'Naïve Bayes', report, 'Rủi ro thấp')

Dự đoán: ['Rủi ro cao', 'Rủi ro thấp', 'Rủi ro thấp']


,Dự đoán: Rủi ro thấp,Dự đoán: Rủi ro cao
Thực tế: Rủi ro thấp,2,0
Thực tế: Rủi ro cao,0,1


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Rủi ro thấp,2,0,0,1.0,1.0,1.0
1,Rủi ro cao,1,0,0,1.0,1.0,1.0
2,Trung bình (macro),,,,1.0,1.0,1.0


Đặng Phan Duy - 6451071010

In [15]:
# Kiểm tra lại F-measure bằng thư viện sklearn
for model, pred in [('Cây ID3', pred_id3_0), ('Cây CART', pred_cart_0), ('Naïve Bayes', pred_bayes_0)]:
    print(model, metrics.f1_score(actual_0, pred, labels=['Rủi ro thấp', 'Rủi ro cao'], average=None, zero_division=0).round(4),
          '- macro:', round(metrics.f1_score(actual_0, pred, labels=['Rủi ro thấp', 'Rủi ro cao'], average='macro', zero_division=0), 4))

Cây ID3 [0.8 0. ] - macro: 0.4
Cây CART [0.8 0. ] - macro: 0.4
Naïve Bayes [1. 1.] - macro: 1.0


Đặng Phan Duy - 6451071010

Nhận xét: kết quả chương trình khớp với tài liệu. Precision, recall của lớp Rủi ro thấp là 67% và 100% với cây quyết định, 100% và 100% với Naïve Bayes. F-measure tương ứng là:
- Cây quyết định (ID3 và CART cho cùng kết quả dự đoán): F = 2 × 0.667 × 1 / (0.667 + 1) = **0.8**
- Naïve Bayes: F = 2 × 1 × 1 / (1 + 1) = **1.0**

Ở nhánh 'Đã kết hôn' của cây CART, hai thuộc tính Thu nhập và Sở hữu nhà có chỉ số Gini bằng nhau. Tài liệu chọn Sở hữu nhà, chương trình chọn Thu nhập, nhưng kết quả dự đoán 3 hồ sơ vẫn giống nhau.

## Bài 1 (mục IV.1)

Dự đoán doanh số bán đồ chơi. Chọn **Cao** là lớp dương (doanh số cao là kết quả doanh nghiệp quan tâm); F-measure của lớp Thấp và giá trị trung bình cũng được tính trong bảng.

In [16]:
cols = ['Loại', 'Số màu', 'Kích thước', 'Chất liệu', 'Doanh số bán']
raw = """Điều khiển|3|Nhỏ|Nhựa PP|Cao
Xếp hình|5|Vừa|Cao su|Thấp
Xếp hình|7|To|Nhựa PP|Thấp
Điều khiển|5|Nhỏ|Cao su|Thấp
Búp bê|3|Vừa|Nhựa PP|Thấp
Điều khiển|5|Vừa|Nhựa PP|Cao
Búp bê|5|To|Nhựa PP|Cao
Điều khiển|7|Vừa|Cao su|Thấp
Xếp hình|7|To|Cao su|Cao
Xếp hình|3|To|Nhựa PP|Thấp
Búp bê|3|Nhỏ|Cao su|Thấp
Xếp hình|3|Nhỏ|Nhựa PP|Cao
Điều khiển|5|To|Cao su|Thấp
Búp bê|5|Vừa|Nhựa PP|Cao
Búp bê|7|To|Nhựa PP|Cao"""
data_1 = pd.DataFrame([line.split('|') for line in raw.splitlines()], columns=cols)
attrs_1 = cols[:-1]
target_1 = cols[-1]
data_1

,Loại,Số màu,Kích thước,Chất liệu,Doanh số bán
0,Điều khiển,3,Nhỏ,Nhựa PP,Cao
1,Xếp hình,5,Vừa,Cao su,Thấp
2,Xếp hình,7,To,Nhựa PP,Thấp
3,Điều khiển,5,Nhỏ,Cao su,Thấp
4,Búp bê,3,Vừa,Nhựa PP,Thấp
5,Điều khiển,5,Vừa,Nhựa PP,Cao
6,Búp bê,5,To,Nhựa PP,Cao
7,Điều khiển,7,Vừa,Cao su,Thấp
8,Xếp hình,7,To,Cao su,Cao
9,Xếp hình,3,To,Nhựa PP,Thấp


Đặng Phan Duy - 6451071010

In [17]:
test_1 = pd.DataFrame([line.split('|') for line in """Búp bê|3|To|Cao su
Xếp hình|5|To|Nhựa PP
Điều khiển|3|Vừa|Cao su""".splitlines()], columns=attrs_1)
actual_1 = ['Thấp', 'Thấp', 'Cao']
test_1['Thực tế'] = actual_1
test_1

,Loại,Số màu,Kích thước,Chất liệu,Thực tế
0,Búp bê,3,To,Cao su,Thấp
1,Xếp hình,5,To,Nhựa PP,Thấp
2,Điều khiển,3,Vừa,Cao su,Cao


Đặng Phan Duy - 6451071010

### Cây quyết định ID3

In [18]:
tree_id3_1 = build_tree(data_1, attrs_1, target_1, 'id3')
print("Information gain của các thuộc tính tại nút gốc:", tree_id3_1['scores'])
show_tree(tree_id3_1, 'id3')

Information gain của các thuộc tính tại nút gốc: {'Loại': 0.0258410375, 'Số màu': 0.0064747672, 'Kích thước': 0.0064747672, 'Chất liệu': 0.1858051629}
Gốc: [Chất liệu] (IG = 0.186)
    Chất liệu = Nhựa PP -> [Loại] (IG = 0.252)
        Loại = Điều khiển -> Cao
        Loại = Xếp hình -> [Kích thước] (IG = 0.918)
            Kích thước = To -> Thấp
            Kích thước = Nhỏ -> Cao
        Loại = Búp bê -> [Số màu] (IG = 0.811)
            Số màu = 3 -> Thấp
            Số màu = 5 -> Cao
            Số màu = 7 -> Cao
    Chất liệu = Cao su -> [Loại] (IG = 0.317)
        Loại = Xếp hình -> [Số màu] (IG = 1.000)
            Số màu = 5 -> Thấp
            Số màu = 7 -> Cao
        Loại = Điều khiển -> Thấp
        Loại = Búp bê -> Thấp


Đặng Phan Duy - 6451071010

In [19]:
pd.DataFrame(find_ties(tree_id3_1), columns=['Nút', 'Các thuộc tính bằng nhau', 'Thuộc tính được chọn'])

,Nút,Các thuộc tính bằng nhau,Thuộc tính được chọn
0,Gốc > Chất liệu = Nhựa PP,"Loại, Số màu",Loại
1,Gốc > Chất liệu = Cao su,"Loại, Số màu, Kích thước",Loại
2,Gốc > Chất liệu = Cao su > Loại = Xếp hình,"Số màu, Kích thước",Số màu


Đặng Phan Duy - 6451071010

In [20]:
pred_id3_1 = [predict_tree(tree_id3_1, r) for _, r in test_1.iterrows()]
cm, report = evaluate(actual_1, pred_id3_1, ['Cao', 'Thấp'])
print("Dự đoán:", pred_id3_1)
display(cm)
display(report)
add_summary('Bài 1 (mục IV.1)', 'Cây ID3', report, 'Cao')

Dự đoán: ['Thấp', 'Thấp', 'Thấp']


,Dự đoán: Cao,Dự đoán: Thấp
Thực tế: Cao,0,1
Thực tế: Thấp,0,2


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Cao,0,0,1,0.000000,0.0,0.0
1,Thấp,2,1,0,0.666667,1.0,0.8
2,Trung bình (macro),,,,0.333333,0.5,0.4


Đặng Phan Duy - 6451071010

### Cây quyết định CART

In [21]:
tree_cart_1 = build_tree(data_1, attrs_1, target_1, 'cart')
print("Gini index của các thuộc tính tại nút gốc:", tree_cart_1['scores'])
show_tree(tree_cart_1, 'cart')

Gini index của các thuộc tính tại nút gốc: {'Loại': 0.48, 'Số màu': 0.4933333333, 'Kích thước': 0.4933333333, 'Chất liệu': 0.3777777778}
Gốc: [Chất liệu] (Gini = 0.378)
    Chất liệu = Nhựa PP -> [Loại] (Gini = 0.315)
        Loại = Điều khiển -> Cao
        Loại = Xếp hình -> [Kích thước] (Gini = 0.000)
            Kích thước = To -> Thấp
            Kích thước = Nhỏ -> Cao
        Loại = Búp bê -> [Số màu] (Gini = 0.000)
            Số màu = 3 -> Thấp
            Số màu = 5 -> Cao
            Số màu = 7 -> Cao
    Chất liệu = Cao su -> [Loại] (Gini = 0.167)
        Loại = Xếp hình -> [Số màu] (Gini = 0.000)
            Số màu = 5 -> Thấp
            Số màu = 7 -> Cao
        Loại = Điều khiển -> Thấp
        Loại = Búp bê -> Thấp


Đặng Phan Duy - 6451071010

In [22]:
pd.DataFrame(find_ties(tree_cart_1), columns=['Nút', 'Các thuộc tính bằng nhau', 'Thuộc tính được chọn'])

,Nút,Các thuộc tính bằng nhau,Thuộc tính được chọn
0,Gốc > Chất liệu = Cao su,"Loại, Số màu, Kích thước",Loại
1,Gốc > Chất liệu = Cao su > Loại = Xếp hình,"Số màu, Kích thước",Số màu


Đặng Phan Duy - 6451071010

In [23]:
pred_cart_1 = [predict_tree(tree_cart_1, r) for _, r in test_1.iterrows()]
cm, report = evaluate(actual_1, pred_cart_1, ['Cao', 'Thấp'])
print("Dự đoán:", pred_cart_1)
display(cm)
display(report)
add_summary('Bài 1 (mục IV.1)', 'Cây CART', report, 'Cao')

Dự đoán: ['Thấp', 'Thấp', 'Thấp']


,Dự đoán: Cao,Dự đoán: Thấp
Thực tế: Cao,0,1
Thực tế: Thấp,0,2


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Cao,0,0,1,0.000000,0.0,0.0
1,Thấp,2,1,0,0.666667,1.0,0.8
2,Trung bình (macro),,,,0.333333,0.5,0.4


Đặng Phan Duy - 6451071010

### Naïve Bayes (làm trơn Laplace)

In [24]:
bayes_1 = predict_bayes(data_1, attrs_1, target_1, test_1)
bayes_1

,Cao,Thấp,Dự đoán
0,0.005020,0.011455,Thấp
1,0.017569,0.010183,Cao
2,0.002824,0.015274,Thấp


Đặng Phan Duy - 6451071010

In [25]:
pred_bayes_1 = list(bayes_1['Dự đoán'])
cm, report = evaluate(actual_1, pred_bayes_1, ['Cao', 'Thấp'])
print("Dự đoán:", pred_bayes_1)
display(cm)
display(report)
add_summary('Bài 1 (mục IV.1)', 'Naïve Bayes', report, 'Cao')

Dự đoán: ['Thấp', 'Cao', 'Thấp']


,Dự đoán: Cao,Dự đoán: Thấp
Thực tế: Cao,0,1
Thực tế: Thấp,1,1


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,Cao,0,1,1,0.00,0.00,0.00
1,Thấp,1,1,1,0.50,0.50,0.50
2,Trung bình (macro),,,,0.25,0.25,0.25


Đặng Phan Duy - 6451071010

In [26]:
# Kiểm tra lại F-measure bằng thư viện sklearn
for model, pred in [('Cây ID3', pred_id3_1), ('Cây CART', pred_cart_1), ('Naïve Bayes', pred_bayes_1)]:
    print(model, metrics.f1_score(actual_1, pred, labels=['Cao', 'Thấp'], average=None, zero_division=0).round(4),
          '- macro:', round(metrics.f1_score(actual_1, pred, labels=['Cao', 'Thấp'], average='macro', zero_division=0), 4))

Cây ID3 [0.  0.8] - macro: 0.4
Cây CART [0.  0.8] - macro: 0.4
Naïve Bayes [0.  0.5] - macro: 0.25


Đặng Phan Duy - 6451071010

Nhận xét:
- Hai cây ID3 và CART cho cùng kết quả dự đoán: cả 3 sản phẩm đều có doanh số Thấp. Vì vậy lớp Cao có precision = recall = 0 nên **F = 0**; lớp Thấp có precision ≈ 0.667, recall = 1 nên **F = 2 × 0.667 × 1 / (0.667 + 1) = 0.8**; trung bình **F ≈ 0.4**.
- Naïve Bayes dự đoán Thấp, Cao, Thấp nên đúng 1/3 sản phẩm: lớp Cao có **F = 0**, lớp Thấp có precision = recall = 0.5 nên **F = 0.5**; trung bình **F = 0.25**.
- Ở nhánh 'Nhựa PP' của cây ID3, hai thuộc tính Loại và Số màu có độ lợi thông tin bằng nhau. Chương trình chọn Loại (giống cây CART, nơi Loại có Gini nhỏ nhất). Nếu chọn Số màu thì sản phẩm thứ 2 (Xếp hình, 5, To, Nhựa PP) sẽ được dự đoán là Cao và kết quả của cây ID3 sẽ thay đổi.

## Bài 2 (mục IV.2)

Phân tích cảm xúc với 3 lớp: tốt, bình thường, xấu. Bài toán có nhiều lớp nên F-measure được tính cho từng lớp (lần lượt xem mỗi lớp là lớp dương) và lấy trung bình (macro). Đề bài chỉ yêu cầu cây CART nên phần này chỉ dùng cây CART và Naïve Bayes.

In [27]:
cols = ['giảm', 'người', 'chuyển', 'yêu', 'vừa', 'đi', 'Cảm xúc']
raw = """0..5|11..20|>20|11..20|>20|0..5|tốt
11..20|6..10|6..10|0..5|11..20|11..20|tốt
6..10|0..5|6..10|11..20|0..5|6..10|xấu
>20|0..5|11..20|6..10|0..5|>20|bình thường
0..5|>20|11..20|0..5|6..10|0..5|xấu
0..5|6..10|0..5|0..5|11..20|11..20|xấu
0..5|6..10|11..20|0..5|6..10|0..5|tốt
11..20|>20|0..5|11..20|0..5|11..20|bình thường
0..5|0..5|6..10|6..10|6..10|>20|tốt
11..20|0..5|11..20|11..20|0..5|11..20|tốt
>20|6..10|0..5|0..5|0..5|6..10|xấu
0..5|0..5|11..20|0..5|11..20|>20|bình thường
6..10|11..20|6..10|>20|0..5|6..10|bình thường
11..20|6..10|>20|11..20|0..5|0..5|xấu"""
data_2 = pd.DataFrame([line.split('|') for line in raw.splitlines()], columns=cols)
attrs_2 = cols[:-1]
target_2 = cols[-1]
data_2

,giảm,người,chuyển,yêu,vừa,đi,Cảm xúc
0,0..5,11..20,>20,11..20,>20,0..5,tốt
1,11..20,6..10,6..10,0..5,11..20,11..20,tốt
2,6..10,0..5,6..10,11..20,0..5,6..10,xấu
3,>20,0..5,11..20,6..10,0..5,>20,bình thường
4,0..5,>20,11..20,0..5,6..10,0..5,xấu
5,0..5,6..10,0..5,0..5,11..20,11..20,xấu
6,0..5,6..10,11..20,0..5,6..10,0..5,tốt
7,11..20,>20,0..5,11..20,0..5,11..20,bình thường
8,0..5,0..5,6..10,6..10,6..10,>20,tốt
9,11..20,0..5,11..20,11..20,0..5,11..20,tốt


Đặng Phan Duy - 6451071010

In [28]:
test_2 = pd.DataFrame([line.split('|') for line in """0..5|6..10|0..5|11..20|6..10|0..5
0..5|0..5|6..10|0..5|11..20|>20
6..10|0..5|11..20|>20|6..10|6..10
6..10|11..20|6..10|6..10|>20|0..5""".splitlines()], columns=attrs_2)
actual_2 = ['xấu', 'tốt', 'bình thường', 'tốt']
test_2['Thực tế'] = actual_2
test_2

,giảm,người,chuyển,yêu,vừa,đi,Thực tế
0,0..5,6..10,0..5,11..20,6..10,0..5,xấu
1,0..5,0..5,6..10,0..5,11..20,>20,tốt
2,6..10,0..5,11..20,>20,6..10,6..10,bình thường
3,6..10,11..20,6..10,6..10,>20,0..5,tốt


Đặng Phan Duy - 6451071010

### Cây quyết định CART

In [29]:
tree_cart_2 = build_tree(data_2, attrs_2, target_2, 'cart')
print("Gini index của các thuộc tính tại nút gốc:", tree_cart_2['scores'])
show_tree(tree_cart_2, 'cart')

Gini index của các thuộc tính tại nút gốc: {'giảm': 0.5833333333, 'người': 0.5428571429, 'chuyển': 0.5738095238, 'yêu': 0.5619047619, 'vừa': 0.5442176871, 'đi': 0.5119047619}
Gốc: [đi] (Gini = 0.512)
    đi = 0..5 -> [người] (Gini = 0.250)
        người = 11..20 -> tốt
        người = >20 -> xấu
        người = 6..10 -> [giảm] (Gini = 0.000)
            giảm = 0..5 -> tốt
            giảm = 11..20 -> xấu
    đi = 11..20 -> [người] (Gini = 0.250)
        người = 6..10 -> [giảm] (Gini = 0.000)
            giảm = 11..20 -> tốt
            giảm = 0..5 -> xấu
        người = >20 -> bình thường
        người = 0..5 -> tốt
    đi = 6..10 -> [người] (Gini = 0.000)
        người = 0..5 -> xấu
        người = 6..10 -> xấu
        người = 11..20 -> bình thường
    đi = >20 -> [chuyển] (Gini = 0.000)
        chuyển = 11..20 -> bình thường
        chuyển = 6..10 -> tốt


Đặng Phan Duy - 6451071010

In [30]:
pd.DataFrame(find_ties(tree_cart_2), columns=['Nút', 'Các thuộc tính bằng nhau', 'Thuộc tính được chọn'])

,Nút,Các thuộc tính bằng nhau,Thuộc tính được chọn
0,Gốc > đi = 0..5,"người, vừa",người
1,Gốc > đi = 0..5 > người = 6..10,"giảm, chuyển, yêu, vừa",giảm
2,Gốc > đi = 11..20,"người, chuyển",người
3,Gốc > đi = 11..20 > người = 6..10,"giảm, chuyển",giảm
4,Gốc > đi = 6..10,"người, yêu",người
5,Gốc > đi = >20,"chuyển, vừa",chuyển


Đặng Phan Duy - 6451071010

In [31]:
pred_cart_2 = [predict_tree(tree_cart_2, r) for _, r in test_2.iterrows()]
cm, report = evaluate(actual_2, pred_cart_2, ['tốt', 'bình thường', 'xấu'])
print("Dự đoán:", pred_cart_2)
display(cm)
display(report)
add_summary('Bài 2 (mục IV.2)', 'Cây CART', report, None)

Dự đoán: ['tốt', 'tốt', 'xấu', 'tốt']


,Dự đoán: tốt,Dự đoán: bình thường,Dự đoán: xấu
Thực tế: tốt,2,0,0
Thực tế: bình thường,0,0,1
Thực tế: xấu,1,0,0


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,tốt,2,1,0,0.666667,1.000000,0.800000
1,bình thường,0,0,1,0.000000,0.000000,0.000000
2,xấu,0,1,1,0.000000,0.000000,0.000000
3,Trung bình (macro),,,,0.222222,0.333333,0.266667


Đặng Phan Duy - 6451071010

### Naïve Bayes (làm trơn Laplace)

In [32]:
bayes_2 = predict_bayes(data_2, attrs_2, target_2, test_2)
bayes_2

,tốt,xấu,bình thường,Dự đoán
0,0.000215,0.000430,0.000009,xấu
1,0.000287,0.000064,0.000162,tốt
2,0.000018,0.000032,0.000081,bình thường
3,0.000048,0.000008,0.000018,tốt


Đặng Phan Duy - 6451071010

In [33]:
pred_bayes_2 = list(bayes_2['Dự đoán'])
cm, report = evaluate(actual_2, pred_bayes_2, ['tốt', 'bình thường', 'xấu'])
print("Dự đoán:", pred_bayes_2)
display(cm)
display(report)
add_summary('Bài 2 (mục IV.2)', 'Naïve Bayes', report, None)

Dự đoán: ['xấu', 'tốt', 'bình thường', 'tốt']


,Dự đoán: tốt,Dự đoán: bình thường,Dự đoán: xấu
Thực tế: tốt,2,0,0
Thực tế: bình thường,0,1,0
Thực tế: xấu,0,0,1


,Lớp,TP,FP,FN,Precision,Recall,F-measure
0,tốt,2,0,0,1.0,1.0,1.0
1,bình thường,1,0,0,1.0,1.0,1.0
2,xấu,1,0,0,1.0,1.0,1.0
3,Trung bình (macro),,,,1.0,1.0,1.0


Đặng Phan Duy - 6451071010

In [34]:
# Kiểm tra lại F-measure bằng thư viện sklearn
for model, pred in [('Cây CART', pred_cart_2), ('Naïve Bayes', pred_bayes_2)]:
    print(model, metrics.f1_score(actual_2, pred, labels=['tốt', 'bình thường', 'xấu'], average=None, zero_division=0).round(4),
          '- macro:', round(metrics.f1_score(actual_2, pred, labels=['tốt', 'bình thường', 'xấu'], average='macro', zero_division=0), 4))

Cây CART [0.8 0.  0. ] - macro: 0.2667
Naïve Bayes [1. 1. 1.] - macro: 1.0


Naïve Bayes [1. 1. 1.] - macro: 1.0


Đặng Phan Duy - 6451071010

Nhận xét:
- Cây CART dự đoán đúng 2/4 trạng thái: lớp tốt có precision ≈ 0.667, recall = 1 nên **F = 0.8**; lớp bình thường và xấu đều có **F = 0**; trung bình **F ≈ 0.267**.
- Naïve Bayes dự đoán đúng cả 4 trạng thái nên precision, recall và **F-measure của cả 3 lớp đều bằng 1**.
- Cây CART của bài này có nhiều nút mà các thuộc tính có chỉ số Gini bằng nhau (bảng ở trên), chương trình chọn thuộc tính đứng trước trong bảng dữ liệu. Ví dụ ở nhánh 'đi = 6..10', nếu chọn 'yêu' thay cho 'người' thì trạng thái thứ 3 sẽ được dự đoán đúng là bình thường. Điều này cho thấy với dữ liệu ít (14 dòng), cây quyết định phụ thuộc nhiều vào cách chọn thuộc tính khi bằng nhau.

## Tổng hợp F-measure của các bài tập cơ bản

In [35]:
pd.DataFrame(summary, columns=['Bài', 'Mô hình', 'Lớp dương', 'Precision', 'Recall',
                              'F-measure (lớp dương)', 'F-measure (trung bình macro)']).round(3)

,Bài,Mô hình,Lớp dương,Precision,Recall,F-measure (lớp dương),F-measure (trung bình macro)
0,Yêu cầu cơ bản (mục III.1),Cây ID3,Rủi ro thấp,0.667,1.000,0.800,0.400
1,Yêu cầu cơ bản (mục III.1),Cây CART,Rủi ro thấp,0.667,1.000,0.800,0.400
2,Yêu cầu cơ bản (mục III.1),Naïve Bayes,Rủi ro thấp,1.000,1.000,1.000,1.000
3,Bài 1 (mục IV.1),Cây ID3,Cao,0.000,0.000,0.000,0.400
4,Bài 1 (mục IV.1),Cây CART,Cao,0.000,0.000,0.000,0.400
5,Bài 1 (mục IV.1),Naïve Bayes,Cao,0.000,0.000,0.000,0.250
6,Bài 2 (mục IV.2),Cây CART,(macro),0.222,0.333,0.267,0.267
7,Bài 2 (mục IV.2),Naïve Bayes,(macro),1.000,1.000,1.000,1.000


Đặng Phan Duy - 6451071010

Nhận xét chung:
- Ở cả 3 bài, F-measure giúp so sánh các mô hình bằng một con số duy nhất thay vì phải xét riêng precision và recall. Ví dụ ở Yêu cầu cơ bản, cây quyết định có recall bằng Naïve Bayes (cùng bằng 1) nhưng precision thấp hơn, nên F-measure thấp hơn (0.8 so với 1.0).
- Naïve Bayes cho F-measure cao hơn hoặc bằng cây quyết định ở Yêu cầu cơ bản và Bài 2, trong khi ở Bài 1 cây quyết định tốt hơn (F trung bình 0.4 so với 0.25).
- Tập kiểm thử của các bài cơ bản rất nhỏ (3 - 4 mẫu) nên chỉ cần thay đổi 1 dự đoán là F-measure thay đổi rất lớn; các kết quả này chỉ mang tính minh họa cách tính, chưa đủ để kết luận mô hình nào tốt hơn.